# Homework Setup

In [1]:
# Finding out the venv it's using.
import sys
print(sys.executable)

/Users/airdedavid/Desktop/Coding/llm-zoomcamp-mavidvd/Homework/02-VectorSearch/.venv/bin/python3


In [2]:
from _ingest import load_data

documents = load_data()

In [3]:
texts = [doc["question"] + " " + doc["answer"] for doc in documents]

In [4]:
from tqdm.auto import tqdm
import numpy as np
from _embedder import Embedder

embed = Embedder()

batch_size = 50
X = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)

  0%|          | 0/27 [00:00<?, ?it/s]

In [5]:
query = "Can I still join the course after the start date?"
v_query = embed.encode(query)

scores = X.dot(v_query)
idx = np.argmax(scores)

documents[idx]

{'id': '3f1424af17',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

# Homework Questions

In [6]:
# 1. What is the first value of the embedded vector?

from _embedder import Embedder

embed = Embedder()

q1 = "How does approximate nearest neighbor search work?"
v1 = embed.encode(q1)
v1[0]

np.float64(-0.020582036807885073)

In [7]:
# 2. Cosine Similarity

# Loading the data.

from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [24]:
documents[:5]

[{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a sim

In [9]:
doc = next(d for d in documents if d['filename'] == '02-vector-search/lessons/07-sqlitesearch-vector.md')
doc

{'content': '# Vector Search with sqlitesearch\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=csxKescwJYM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous section we used minsearch for vector search.\n\nIt works, but it has three problems:\n\n- It rebuilds the index on every startup\n- It keeps everything in memory\n- It searches by brute force\n\n\nWith text search we never felt these. Indexing was fast because we\ndidn\'t embed anything. With vector search, indexing runs a neural\nnetwork over every document, so it takes a minute on our dataset.\nKeeping everything in memory is fine here, but a larger dataset would\nneed too much space.\n\nThe third problem is brute-force search. For every query we compare the\nquery vector against every single document. With 1,000 documents this is\nfine, probably even faster than anything smarter. But as the dataset\ngrows past 10,000 or so, it gets slow, and we\'ll want an approximate\nmethod instead.\n\nWhat we\'ve done 

In [10]:
vdoc = embed.encode(doc['content'])

In [11]:
vdoc.dot(v1)

np.float64(0.361070280302606)

In [25]:
# 3. Chunking and search by hand.

from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)
chunks[0]

{'start': 0,
 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phon

In [13]:
contents = [c['content'] for c in chunks]

X = embed.encode_batch(contents)

In [14]:
scores = X.dot(v1)

In [15]:
best_idx = np.argmax(scores)
best_idx

np.int64(94)

In [16]:
print(chunks[best_idx])

{'start': 1000, 'content': 'rch. We score\nthe query against every document and pick the top ones. It always finds\nthe true top matches, but it pays for that by touching everything.\n\nApproximate nearest neighbor (ANN) search takes a shortcut. Instead of\ncomparing against everything, it first narrows down to a region of\nlikely matches. Then it scores only within that region. It may miss the\nabsolute best match, but the results are still good and it\'s much\nfaster.\n\n```text\nNN (exact):    compare query against ALL documents -> top 5\nANN (approx):  narrow down to a region -> compare within region -> top 5\n```\n\n## sqlitesearch\n\nsqlitesearch is the persistent sibling of minsearch, and it solves both\nproblems at once.\n\nWe already used it in module 1 for persistent text search. It also does\nvector search through its `VectorSearchIndex` class. It stores vectors\nin SQLite, a real on-disk database, and uses ANN strategies for\nretrieval. Because the data lives on disk, one p

In [17]:
# 4. Vector search with minsearch

from minsearch import VectorSearch
import numpy as np

q2 = 'What metric do we use to evaluate a search engine?'
v2 = embed.encode(q2)

# To match the results we need to take documents, not chunks.
contents = [c['content'] for c in documents]
X = embed.encode_batch(contents)

vindex = VectorSearch()
vindex.fit(X, documents)

results = vindex.search(v2, num_results=1)

In [18]:
results[0]['filename']

'04-evaluation/lessons/05-search-metrics.md'

In [19]:
# 5. Keyword Search vs Vector Search.

from minsearch import Index

q = 'How do I store vectors in PostgreSQL?'

kindex = Index(text_fields=['content'], keyword_fields=[])
kindex.fit(documents)

In [20]:
k_results = kindex.search(q, num_results = 5)
k_results

[{'content': '# Embeddings\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=kJOlW1HeMp4&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nBefore we can do vector search, we need to turn our text into vectors.\nWe call this process embedding: we embed text into a vector space. The\nvectors we get back are also called "embeddings."\n\n## Word embeddings and sentence embeddings\n\nThis idea comes from\n[word2vec](https://en.wikipedia.org/wiki/Word2vec). The model learns to\nplace words as points in a multi-dimensional space. Words with similar\nmeanings land close to each other.\n\nImagine a 2D space where "enroll" and "join" are near each other and\n"Docker" is far away:\n\n```text\n        · enroll\n       · join\n                   · Docker\n```\n\nThe same idea works for entire sentences:\n\n```text\nQ1: "I just discovered the course. Can I still join it?"\nQ2: "I just found out about the program. Can I still enroll?"\n\nThese two are close - they mean the same thing.\n\nQ3: "H

In [21]:
vq = embed.encode(q)
v_results = vindex.search(vq, num_results = 5)
v_results

[{'content': '# Vector Search with PGVector\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=0P54MFyz-mc&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nMany real databases can do vector search. Elasticsearch has it, and\nthere are dedicated stores like Qdrant and Chroma. We\'ll go with\nPostgres. Most of us already run it at work, and the data engineering\ncourse uses it too. The concept is the same as with sqlitesearch, only\nthe database under the hood changes.\n\n[pgvector](https://github.com/pgvector/pgvector) is the PostgreSQL\nextension that makes this work. Install it and Postgres can do vector\nsimilarity search. On top of that you get the usual production features,\nlike concurrent access, transactions, and large datasets.\n\nWe\'ll run Postgres with pgvector in Docker.\n\n## Starting Postgres with pgvector\n\nPull the image and start a container:\n\n```bash\ndocker run -it \\\n    --name pgvector \\\n    -e POSTGRES_USER=user \\\n    -e POSTGRES_PASSWORD=pswd \\\n  

In [22]:
# 6. Hybrid Search: Reciprocal Rank Fusion

q2 = "How do I give the model access to tools?"

kindex = Index(text_fields=['content'], keyword_fields=[])
kindex.fit(documents)

k2_results = kindex.search(q2)
k2_results

vq2 = embed.encode(q2)
v2_results = vindex.search(vq2)
v2_results

def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["content"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [23]:
rrf_results = rrf([v2_results, k2_results])
rrf_results

[{'content': "# Other Frameworks\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=4yiCbKX9RhI&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nThe concepts you learned in Part 2 are the same across frameworks.\nFunction calling, the agent loop, and tool definitions all wrap the\nsame pattern. Send messages, run any function calls, and repeat until\nthe model is done.\n\nYou now understand how the loop works. So you can pick up any\nproduction framework and know what it's doing under the hood. I kept\nthis module framework agnostic on purpose, so you can explore and pick\nthe one you like.\n\nHere are some frameworks worth a look:\n\n## OpenAI Agents SDK\n\nThe official SDK from OpenAI for building agents. It uses the same\nResponses API we used throughout this module. It supports tool\ndefinition, multi-turn conversations, and handoffs between agents.\n\n```bash\nuv add openai-agents\n```\n\nGood choice if you're already using OpenAI and want something\nofficial and well-mainta